# Interactive K-Means Steps (Lecture 11)

Choose initial centroids and step through assignment/update while varying the number of clusters `k`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))

centers = np.array([[-2.2, -1.4], [2.2, -1.0], [0.3, 2.2]])


def make_data(n=150, noise=0.45, seed=6):
    rng = rng_from_seed(seed)
    per = n // len(centers)
    pts = []
    for c in centers:
        pts.append(c + noise * rng.normal(size=(per, 2)))
    X = np.vstack(pts)
    while len(X) < n:
        c = centers[len(X) % len(centers)]
        X = np.vstack([X, c + noise * rng.normal(size=(1, 2))])
    return X


def assign_labels(X, C):
    d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    return d.argmin(axis=1)


def update_centroids(X, labels, C):
    new_c = []
    for j in range(len(C)):
        pts = X[labels == j]
        new_c.append(pts.mean(axis=0) if len(pts) else C[j])
    return np.vstack(new_c)


def draw(k=3, n=150, noise=0.45, seed=6, step="assign"):
    X = make_data(n=n, noise=noise, seed=seed)
    rng = rng_from_seed(1000 + seed)
    picks = rng.choice(len(X), size=k, replace=False)
    C = X[picks]
    labels = np.full(len(X), -1)
    if step in {"assign", "iterate"}:
        labels = assign_labels(X, C)
    if step == "iterate":
        C = update_centroids(X, labels, C)
        labels = assign_labels(X, C)
    fig, axs = plt.subplots(1, 2, figsize=(12.8, 4.6))
    axs[0].scatter(X[:, 0], X[:, 1], c=("0.78" if step == "choose" else labels), s=18, cmap="tab10")
    axs[0].scatter(C[:, 0], C[:, 1], c="k", marker="x", s=180, linewidths=3)
    axs[0].set_title("K-means step walkthrough")
    axs[0].grid(alpha=0.2)
    has_assignments = np.any(labels >= 0)
    counts = np.bincount(labels[labels >= 0], minlength=k) if has_assignments else np.zeros(k, dtype=int)
    axs[1].bar(range(k), counts, color=("#2563eb" if has_assignments else "#cbd5e1"))
    axs[1].set_xticks(range(k), [f"C{i}" for i in range(k)])
    axs[1].set_title("Cluster sizes")
    axs[1].set_ylim(0, max(1, len(X)))
    if not has_assignments:
        axs[1].text(0.5, 0.6, "Choose centroids, then run Assignment step", transform=axs[1].transAxes, ha="center", va="center", color="#475569")
    axs[1].grid(alpha=0.2)
    plt.show()

widgets.interact(
    draw,
    k=widgets.IntSlider(min=2, max=6, step=1, value=3, continuous_update=False),
    n=widgets.IntSlider(min=60, max=240, step=30, value=150, continuous_update=False),
    noise=widgets.FloatSlider(min=0.15, max=1.1, step=0.05, value=0.45, continuous_update=False),
    seed=widgets.IntSlider(min=1, max=20, step=1, value=6, continuous_update=False),
    step=widgets.Dropdown(options=["choose", "assign", "iterate"], value="assign"),
)
